In [1]:
import pandas as pd
from utils import ngram_utils, gpt2_utils, grnn_utils
from utils.text_utils import *

Loading the LM will be faster if you build a binary file.
Reading /home/marrsia/UCL_dissertation/ucl-fgd/models/ngram/model.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
/home/marrsia/.local/lib/python3.8/site-packages/transformers/modeling_utils.py:415: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they ar

## Generating samples for labeling
Not using ngram, because the outputs are all gibberish: ngrams aren't relly well suited for autorecursive generation.

In [3]:
sentence_starts_dict = build_sentence_starts_for_sampling("../data/stimuli/pilot_subject_gap.csv", gap_type='subject')

In [4]:
for sentence_id, entry in sentence_starts_dict.items():
    sentence_start = entry['sentence_start']
    
    entry['gpt2_continuations'] = [
        gpt2_utils.sample_continuation(sentence_start) 
        for _ in range(5)
    ]
    entry['grnn_continuations'] = [
        grnn_utils.sample_continuation(sentence_start) 
        for _ in range(5)
    ]

In [5]:
import spacy
nlp = spacy.load("en_core_web_sm")

for sentence_id, entry in sentence_starts_dict.items():
    sentence_start = entry['sentence_start']
    n_start_tokens = len(nlp(sentence_start))
    
    entry['gpt2_pos'] = []
    for continuation in entry['gpt2_continuations']:
        doc = nlp(sentence_start + ' ' + continuation)
        entry['gpt2_pos'].append([(token.text, token.pos_) for token in doc[n_start_tokens:]])
    
    entry['grnn_pos'] = []
    for continuation in entry['grnn_continuations']:
        doc = nlp(sentence_start + ' ' + continuation)
        entry['grnn_pos'].append([(token.text, token.pos_) for token in doc[n_start_tokens:]])

/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:36: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  hasattr(torch, "has_mps")
/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:37: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  and torch.has_mps  # type: ignore[attr-defined]


In [6]:
import pandas as pd

rows = []
for sentence_id, entry in sentence_starts_dict.items():
    base = {
        'sentence_id': sentence_id,
        'condition': entry['condition'],
        'embedding_level': entry['levels_of_embedding'],
        'sentence_start': entry['sentence_start'],
    }
    
    for i, (continuation, pos_tags) in enumerate(zip(entry['gpt2_continuations'], entry['gpt2_pos'])):
        rows.append({
            **base,
            'model': 'gpt2',
            'continuation_start_pos': pos_tags[0][1] if pos_tags else None,
            'continuation': continuation,
        })
    
    for i, (continuation, pos_tags) in enumerate(zip(entry['grnn_continuations'], entry['grnn_pos'])):
        rows.append({
            **base,
            'model': 'grnn',
            'continuation_start_pos': pos_tags[0][1] if pos_tags else None,
            'continuation': continuation,
        })

df = pd.DataFrame(rows)
cols = ['sentence_id', 'condition', 'embedding_level', 'model', 'continuation_start_pos', 'sentence_start', 'continuation']
df = df[cols]

df.to_csv('../data/subject_continuations_to_label.csv', index=False)